# MutuneRent Pro - Google Colab Parallel GPU Pipeline

This notebook trains a 3D Gaussian Splatting model from a set of images captured using the Mutune Guided 360 Photo Capture tool.
Run this notebook on a T4 or A100 GPU instance in Google Colab.

In [ ]:
!nvidia-smi

## 1. Setup Environment

In [ ]:
!pip install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118
!pip install plyfile tqdm pillow
!apt-get install -y colmap

## 2. Clone Gaussian Splatting Repo

In [ ]:
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd gaussian-splatting
!pip install -e submodules/diff-gaussian-rasterization
!pip install -e submodules/simple-knn

## 3. Download and Prepare Dataset

Set the `SCAN_URLS` list to the URLs of the uploaded photos.

In [ ]:
import os
import urllib.request

SCAN_URLS = [
    # Add image URLs here
]

INPUT_DIR = "/content/dataset/input"
os.makedirs(INPUT_DIR, exist_ok=True)

for i, url in enumerate(SCAN_URLS):
    print(f"Downloading {i+1}/{len(SCAN_URLS)}...")
    urllib.request.urlretrieve(url, os.path.join(INPUT_DIR, f"img_{i:04d}.jpg"))

## 4. Run COLMAP

In [ ]:
!python convert.py -s /content/dataset

## 5. Train Model

In [ ]:
!python train.py -s /content/dataset -m /content/output --iterations 7000

## 6. Download Result

In [ ]:
try:
    from google.colab import files
    files.download('/content/output/point_cloud/iteration_7000/point_cloud.ply')
except ImportError:
    print('Not in Colab, skip downloading.')